In [29]:
# Ensure packages are installed properly
import sys
print(sys.executable)

/Users/rjiminian/.local/share/virtualenvs/claudeApiProject-1xE5Hto1/bin/python


In [7]:
# Load env variables
from dotenv import load_dotenv
load_dotenv()

True

In [8]:
# Create an API Client
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-5"

In [26]:
# Helper Functions
def add_user_message(messages, text):
  user_message = { "role": "user", "content": text}
  messages.append(user_message)

def add_assistant_message(messages, text):
  assistant_message = { "role": "assistant", "content": text}
  messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
  params = {
    "model": model,
    "max_tokens": 1000,
    "messages": messages,
    "temperature": temperature
  }

  if system:
    params["system"] = system

  if stop_sequences:
    params["stop_sequences"] = stop_sequences

  message = client.messages.create(**params)

  return message.content[0].text

In [13]:
# Make a starting list of messages
messages = []

# Add in the initial user question of "Define quantum computing in one sentence"
add_user_message(messages, "Define quantum computing in one sentence.")

# Pass the list of messages into "chat" to get an answer
answer = chat(messages)
print(answer)

# Take the answer and add it as an assistant message into our list
add_assistant_message(messages, answer)

# Add the user's follow-up question
add_user_message(messages, "Write another sentence.")

# Call chat again with the list of messages to get a final answer
answer = chat(messages)

print(answer)

Quantum computing is a type of computation that harnesses quantum mechanical phenomena like superposition and entanglement to process information in ways that can solve certain problems exponentially faster than classical computers.
Quantum computers use quantum bits (qubits) that can exist in multiple states simultaneously, unlike classical bits which are either 0 or 1.


In [ ]:
messages = []

while True:
  user_prompt = input("> ")
  print(">", user_prompt)

  add_user_message(messages, user_prompt)

  answer = chat(messages)

  add_assistant_message(messages, answer)

  print("---")
  print(answer)
  print("---")

What is 1 + 1?
1 + 1 = 2
Add 5 to that
2 + 5 = 7



BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.4: user messages must have non-empty content'}, 'request_id': 'req_011CbU3jkinVYrs3Fg64D6BM'}

In [16]:
messages = []

system = """
  You are a patient math tutor.
  Do not directly answer a student's questions.
  Guide them to a solution step by step.
"""

add_user_message(messages, "How do I solve 5x + 3 = 2 for x?")

answer = chat(messages, system=system)

print(answer)

Great question! Let's work through this together step by step.

When we have an equation like **5x + 3 = 2**, our goal is to get x by itself on one side. 

**First, let me ask you:** What do you notice is "with" the x term on the left side of the equation? What operation is connecting it to the 5x?

Once you identify that, think about what we could do to both sides of the equation to start isolating the x term.


In [17]:
# Exercise
messages = []

add_user_message(messages, "Write a Python function that checks a string for duplicate characters.")

system = """
  You are a Python engineer who writes very concise code.
"""

answer = chat(messages, system=system)

print(answer)

Here's a concise function to check for duplicate characters:

```python
def has_duplicates(s: str) -> bool:
    return len(s) != len(set(s))
```

**Usage:**
```python
print(has_duplicates("hello"))    # True (l appears twice)
print(has_duplicates("world"))    # False
print(has_duplicates(""))         # False
print(has_duplicates("aabbcc"))   # True
```

**How it works:** Converts the string to a set (which removes duplicates), then compares lengths. If they differ, duplicates exist.


In [ ]:
# Temperature coefficient (0-1)
# 0 means deterministic (no guessing, just facts) and 1 is more creative
messages = []

add_user_message(messages, "Generate a one sentence movie idea")

answer = chat(messages, temperature=1.0)

print(answer)

A retired astronaut discovers that the recurring nightmares she's had since returning from Mars are actually suppressed memories of an alien encounter that's about to repeat itself on Earth.


In [22]:
# Streaming (manual)
messages = []

add_user_message(messages, "Write a 1 sentence description of a fake database.")

stream = client.messages.create(
  model=model,
  max_tokens=1000,
  messages=messages,
  stream=True
)

for event in stream:
  print(event)

RawMessageStartEvent(message=Message(id='msg_01FVQTAJW9vmrbmxd2EYngVj', container=None, content=[], model='claude-sonnet-4-5-20250929', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=19, output_tokens=1, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='A', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' fictional customer database containing 50,000 records with names, email addresses, purchase history, and loyalty', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEve

In [ ]:
# Streaming (optimized)
messages = []

add_user_message(messages, "Write a 1 sentence description of a fake database.")

with client.messages.stream(
  model=model,
  max_tokens=1000,
  messages=messages
) as stream:
  for text in stream.text_stream:
    print(text, end="")

# stream.get_final_message() # another way to get the completed message

A fictional database called "MemoryVault" that stores every dream humanity has ever had, indexed by emotion, recurring symbols, and the sleeper's geographical location.

In [28]:
# Assistant message prefilling + stop sequences
# this technique will return only what we are looking for (no headers, explanations, etc.)

messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])

print(text)


{
  "source": ["aws.ec2"],
  "detail-type": ["EC2 Instance State-change Notification"],
  "detail": {
    "state": ["running"]
  }
}

